In [7]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
import seaborn as sns
import cv2
import random
import datetime

In [8]:
from keras.models import Sequential, Model, load_model
from keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,Input,Activation,GlobalAveragePooling2D, BatchNormalization,Reshape
from keras.optimizers import Adam, RMSprop
from keras.layers import LeakyReLU,Conv2DTranspose
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from keras.utils import plot_model
from keras.datasets.cifar10 import load_data
from keras.preprocessing.image import ImageDataGenerator

In [9]:
def define_discriminator(in_shape=(224,224,3)):
    model = Sequential()
    
    model.add(Conv2D(64, (3,3), padding='same', input_shape=in_shape))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(256, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # classifier
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1,activation='sigmoid'))
    opt = Adam(lr=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])
    
    return model

def define_generator(latent_dim):
    model = Sequential()

    n_nodes = 256 * 7 * 7
    model.add(Dense(n_nodes,input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((7,7,256)))
    # upsample to 14x14
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 28x28
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 56x56
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 112x112
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 224x224
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2D(3,(3,3),activation='tanh',padding='same'))
    return model

def define_gan(g_model,d_model):
    d_model.trainable = False
    model = Sequential()
    model.add(g_model)
    model.add(d_model)
    opt = Adam(lr=0.0002,beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt)
    return model

In [10]:
def load_real_samples():
    # load cifar10 dataset
    (trainX,_) , (_,_) = load_data()
    X = trainX.astype('float32')
    # normalize to [-1,1]
    X = (X - 127.5) / 127.5
    return X

def load_dog_and_cat():
    datagen = ImageDataGenerator(rescale=1./255)
    train_generator = datagen.flow_from_directory('./dataset/test_set/', target_size=(224,224), batch_size=32, class_mode='binary')
    # batch = train_generator.next()
    # print(batch[0].shape)
    # labels = train_generator.labels
    # print(labels)
    # set the number of samples in train_generator
    n_samples = len(train_generator.filenames)//train_generator.batch_size * train_generator.batch_size
    images = np.zeros((n_samples,*train_generator.image_shape))
    # images = np.zeros((n_samples, 224, 224, 3), dtype=np.float32)

    labels = np.zeros((n_samples,), dtype=np.float32)
    for i,(x_batch,y_batch) in enumerate(train_generator):
        start_idx = i * train_generator.batch_size
        end_idx = start_idx + train_generator.batch_size
        images[start_idx:end_idx] = x_batch
        labels[start_idx:end_idx] = y_batch
        if end_idx >= n_samples:
            break
        if i * train_generator.batch_size >= n_samples:
            break
    print(images.shape)
    print(labels.shape)
    X = images
    X = (X - 127.5) / 127.5
    return X
    # return images,labels
    # return train_generator
# def load_dog_and_cat():
#     datagen = ImageDataGenerator(rescale=1./255)
#     train_generator = datagen.flow_from_directory('./dataset/test_set/', target_size=(224,224), batch_size=32, class_mode='binary')
#     n_samples = len(train_generator.filenames)
#     images = np.zeros((n_samples, 224, 224, 3), dtype=np.float32)
#     labels = np.zeros((n_samples,), dtype=np.int)
#     i = 0
#     for x_batch, y_batch in train_generator:
#         start_idx = i * train_generator.batch_size
#         end_idx = start_idx + x_batch.shape[0]
#         images[start_idx:end_idx] = x_batch
#         labels[start_idx:end_idx] = y_batch
#         i += 1
#         if i * train_generator.batch_size >= n_samples:
#             break
#     X = (images - 127.5) / 127.5
#     return X

def generate_real_samples(dataset,n_samples):
    # choose random instances
    ix = np.random.randint(0,dataset.shape[0],n_samples)
    # retrieve selected images
    X = dataset[ix]
    # generate 'real' class labels (1)
    y = np.ones((n_samples,1))
    return X,y


def generate_latent_points(latent_dim,n_samples):
    # generate points in the latent space
    x_input = np.random.randn(latent_dim*n_samples)
    # reshape
    x_input = x_input.reshape(n_samples,latent_dim)
    
    return x_input

def generate_fake_samples(g_model,latent_dim,n_samples):
    # generate points in latent space
    x_input = generate_latent_points(latent_dim,n_samples)
    # predict outputs
    X = g_model.predict(x_input)
    # create 'fake' class labels (0)
    y = np.zeros((n_samples,1))
    return X,y


def save_plot(examples,epoch,n=10):
    # plot images
    for i in range(n*n):
        plt.subplot(n,n,1+i)
        plt.axis('off')
        plt.imshow(examples[i,:,:,0],cmap='gray_r')
    # save plot to file
    filename = 'generated_plot_e%03d.png' % (epoch+1)
    plt.savefig(filename)
    plt.close()


def summarize_performance(epoch,g_model,d_model,dataset,latent_dim,n_samples=100):
    X_real,y_real = generate_real_samples(dataset,n_samples)
    _,acc_real = d_model.evaluate(X_real,y_real,verbose=0)
    X_fake,y_fake = generate_fake_samples(g_model,latent_dim,n_samples)
    _,acc_fake = d_model.evaluate(X_fake,y_fake,verbose=0)
    print('Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100,acc_fake*100))
    save_plot(X_fake,epoch)
    filename = 'generator_model_%03d.h5' % (epoch+1)
    g_model.save(filename)


def train(g_model,d_model,gan_model,dataset,latent_dim,n_epochs=10,n_batch=16):
    bat_per_epo = int(dataset.shape[0]/n_batch)
    half_batch = int(n_batch/2)
    for i in range(n_epochs):
        for j in range(bat_per_epo):
            # get randomly selected 'real' samples
            X_real,y_real = generate_real_samples(dataset,half_batch)
            # update discriminator model weights on real samples
            d_loss1,_ = d_model.train_on_batch(X_real,y_real)
            # generate 'fake' examples
            X_fake,y_fake = generate_fake_samples(g_model,latent_dim,half_batch)
            # update discriminator model weights on fake samples
            d_loss2,_ = d_model.train_on_batch(X_fake,y_fake)
            # prepare points in latent space as input for the generator
            X_gan = generate_latent_points(latent_dim,n_batch)
            # create inverted labels for the fake samples
            y_gan = np.ones((n_batch,1))
            # update the generator via the discriminator's error
            g_loss = gan_model.train_on_batch(X_gan,y_gan)
            # summarize loss on this batch
            print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' % (i+1,j+1,bat_per_epo,d_loss1,d_loss2,g_loss))

        if (i+1) % 10 == 0:
            summarize_performance(i,g_model,d_model,dataset,latent_dim)



In [11]:
dataset = load_dog_and_cat()

Found 2000 images belonging to 2 classes.
(1984, 224, 224, 3)
(1984,)


In [12]:
latent_dim = 100

d_model = define_discriminator()
g_model = define_generator(latent_dim)
gan_model = define_gan(g_model,d_model)
train(g_model,d_model,gan_model,dataset,latent_dim)

1/1 [==============================] - 1s 691ms/step
>1, 1/124, d1=0.663, d2=0.696 g=0.696
1/1 [==============================] - 0s 22ms/step
>1, 2/124, d1=0.126, d2=0.691 g=0.707
1/1 [==============================] - 0s 31ms/step
>1, 3/124, d1=0.005, d2=0.676 g=0.729
1/1 [==============================] - 0s 21ms/step
>1, 4/124, d1=0.001, d2=0.653 g=0.767
1/1 [==============================] - 0s 26ms/step
>1, 5/124, d1=0.000, d2=0.616 g=0.843
1/1 [==============================] - 0s 21ms/step
>1, 6/124, d1=0.000, d2=0.557 g=1.010
1/1 [==============================] - 0s 34ms/step
>1, 7/124, d1=0.000, d2=0.435 g=1.469
1/1 [==============================] - 0s 22ms/step
>1, 8/124, d1=0.035, d2=0.326 g=1.804
1/1 [==============================] - 0s 25ms/step
>1, 9/124, d1=0.000, d2=0.277 g=3.324
1/1 [==============================] - 0s 29ms/step
>1, 10/124, d1=0.000, d2=0.018 g=5.309
1/1 [==============================] - 0s 25ms/step
>1, 11/124, d1=9.766, d2=0.440 g=0.573
1/1 [==